In [ ]:
import cv2
import numpy as np
import random

img1_color = cv2.imread('book.png')
img2_color = cv2.imread('books.jpg')

img1_gray = cv2.cvtColor(img1_color, cv2.COLOR_BGR2GRAY)
img2_gray = cv2.cvtColor(img2_color, cv2.COLOR_BGR2GRAY)

orb = cv2.ORB_create(2000)

kp1, des1 = orb.detectAndCompute(img1_gray, None)
kp2, des2 = orb.detectAndCompute(img2_gray, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING)
matches = bf.knnMatch(des1, des2, k=2)

good = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good.append(m)

print(f"Good matches: {len(good)}")

h1, w1 = img1_color.shape[:2]
h2, w2 = img2_color.shape[:2]

canvas_h = max(h1, h2)
canvas_w = w1 + w2

canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
canvas[:h1, :w1] = img1_color
canvas[:h2, w1:w1 + w2] = img2_color

for match in good:
    pt1 = (int(kp1[match.queryIdx].pt[0]),
           int(kp1[match.queryIdx].pt[1]))

    pt2 = (int(kp2[match.trainIdx].pt[0]) + w1,
           int(kp2[match.trainIdx].pt[1]))

    color = (
        random.randint(50, 255),
        random.randint(50, 255),
        random.randint(50, 255)
    )

    cv2.line(canvas, pt1, pt2, color, 1, cv2.LINE_AA)
    cv2.circle(canvas, pt1, 3, color, -1)
    cv2.circle(canvas, pt2, 3, color, -1)

if len(good) > 10:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    if M is not None:
        h, w = img1_gray.shape
        pts = np.float32([[0, 0], [0, h], [w, h], [w, 0]]).reshape(-1, 1, 2)
        dst = cv2.perspectiveTransform(pts, M)

        dst_offset = dst + np.float32([[[w1, 0]]])

        cv2.polylines(
            canvas,
            [np.int32(dst_offset)],
            isClosed=True,
            color=(0, 255, 0),
            thickness=3,
            lineType=cv2.LINE_AA
        )

        cv2.putText(
            canvas,
            f"Matches: {len(good)}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )
else:
    print("Good matches 부족.")

cv2.imwrite('result_matches.png', canvas)
cv2.imshow('Feature Matching Result', canvas)
cv2.waitKey(0)
cv2.destroyAllWindows()


Good matches: 126
